# 🔬 Autonomous AI Researcher — Ablation Study (Kaggle)

**Qwen2.5-14B chạy LOCAL trên Kaggle GPU — load 1 lần, dùng cho cả 6 experiments.**

| Exp | Cấu hình | Mục đích |
|-----|----------|----------|
| 1 | Vanilla LLM (zero-shot) | Baseline thấp nhất |
| 2 | RAG truyền thống | Baseline có truy hồi |
| 3 | Agent (chưa fine-tune) | Đo lợi ích agentic flow |
| 4 | Agent + Reviewer LoRA | Đo đóng góp Reviewer FT |
| 5 | Agent + Reranker FT | Đo đóng góp Reranker FT |
| 6 | Agent đầy đủ (cả hai) | Hệ thống hoàn chỉnh |

**Kaggle Inputs cần add (sidebar phải → Add Data):**
1. `autonomous-researcher-code` — `src_code.zip`
2. `qwen25-14b-instruct` — Qwen2.5-14B model weights (backbone cho cả 4 agent)
3. `reviewer-lora` — (cho Exp4, Exp6)
4. `reranker-ft` — (cho Exp5, Exp6)

**Lưu ý VRAM:** Qwen2.5-14B @ 4-bit ≈ 9GB. Kaggle T4 = 15GB. Chỉ load model 1 lần duy nhất.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 1: Cài đặt thư viện
# ═══════════════════════════════════════════════════════════════
import subprocess, sys

pkgs = [
    "langgraph",
    "sentence-transformers",
    "chromadb",
    "trafilatura",
    "duckduckgo-search",
    "openai",
    "peft",
    "bitsandbytes",
    "accelerate",
    "PyYAML",
    "pandas",
    "matplotlib",
    "python-dotenv",
]

print("📦 Cài đặt thư viện...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)
print("✅ Xong!")

import os
import warnings
import logging

# 1. Khóa cảnh báo ở cấp độ cấu hình môi trường của hệ điều hành (Mạnh nhất)
os.environ['PYTHONWARNINGS'] = 'ignore'

# 2. Tắt toàn bộ cảnh báo trong luồng thực thi Python
warnings.filterwarnings('ignore')
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

# 3. Ép hệ thống logging bắt các cảnh báo "vượt ngục" và hạ cấp độ hiển thị xuống ERROR
logging.captureWarnings(True)
logging.getLogger("py.warnings").setLevel(logging.ERROR)
logging.getLogger("urllib3").setLevel(logging.ERROR)

print("🛡️ Đã kích hoạt chế độ im lặng tuyệt đối! Tất cả các Warning đã bị triệt tiêu.")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 2: Setup working directory (Tối ưu theo cơ chế tự giải nén của Kaggle)
# ═══════════════════════════════════════════════════════════════
import os, sys, shutil

WORKING_DIR = "/kaggle/working"
INPUT_DIR = "/kaggle/input/datasets/ziangtran123/autonomous-res-code"

# Kiểm tra fallback nếu đổi tên dataset
if not os.path.exists(INPUT_DIR):
    for base in ["/kaggle/input/autonomous-researcher-code", "/kaggle/input/autonomous-researcher"]:
        if os.path.exists(base):
            INPUT_DIR = base
            break

print(f"📁 Thư mục nguồn từ Kaggle: {INPUT_DIR}")

# Tiến hành copy toàn bộ file và thư mục con sang /kaggle/working
if os.path.exists(INPUT_DIR):
    print("🚚 Đang copy toàn bộ source code sang thư mục làm việc...")
    for item in os.listdir(INPUT_DIR):
        src = os.path.join(INPUT_DIR, item)
        dst = os.path.join(WORKING_DIR, item)
        if os.path.isdir(src):
            shutil.copytree(src, dst, dirs_exist_ok=True)
        else:
            shutil.copy2(src, dst)
    print("✅ Đã copy xong!")
else:
    raise FileNotFoundError("❌ Không tìm thấy thư mục nguồn! Hãy check lại phần Add Data ở sidebar.")

# Chuyển môi trường làm việc về /kaggle/working
os.chdir(WORKING_DIR)
sys.path.insert(0, WORKING_DIR)

# Tạo các thư mục đầu ra cho thí nghiệm
for d in ["logs/traces/ablation", "data/raw_outputs", "data/benchmarks", "logs/experiments"]:
    os.makedirs(d, exist_ok=True)

print(f"📍 Working dir hiện tại: {os.getcwd()}")
print("📁 Danh sách file/folder tại root:", sorted([f for f in os.listdir(WORKING_DIR) if not f.startswith('.')]))

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 3: Tìm đường dẫn tất cả model inputs từ Kaggle
# ═══════════════════════════════════════════════════════════════
import os

def find_dir(*candidates):
    for p in candidates:
        if os.path.isdir(p):
            return p
    return None

# ── Qwen 14B (backbone chính cho cả 4 agents) ──────────────────
QWEN_MODEL_PATH = find_dir(
    "/kaggle/input/models/qwen-lm/qwen2.5/transformers/14b-instruct/1",
)
if not QWEN_MODEL_PATH:
    raise FileNotFoundError(
        "❌ Không tìm thấy Qwen2.5-14B!\n"
        "Hãy add Qwen2.5-14B vào Kaggle Input với slug 'qwen25-14b-instruct'.\n"
        "Có thể thêm từ: Datasets → Add Data → Search 'qwen2.5 14b instruct'"
    )
print(f"✅ Qwen model    : {QWEN_MODEL_PATH}")

# ── Reviewer LoRA adapter (cho Exp4, Exp6) ────────────────────
REVIEWER_ADAPTER_PATH = find_dir(
    "/kaggle/input/models/ziangtran123/reviewer-lora/transformers/default/1",
)
print(f"{'✅' if REVIEWER_ADAPTER_PATH else '⚠️'} Reviewer LoRA  : {REVIEWER_ADAPTER_PATH or 'Không tìm thấy — Exp4/6 tắt reviewer FT'}")

# ── Reranker CrossEncoder (cho Exp5, Exp6) ────────────────────
RERANKER_PATH = find_dir(
    "/kaggle/input/models/ziangtran123/reranker-ft/transformers/default/1",
)
print(f"{'✅' if RERANKER_PATH else '⚠️'} Reranker       : {RERANKER_PATH or 'Không tìm thấy — Exp5/6 tắt reranker'}")

print("\n📊 VRAM Budget Estimate:")
print("  Qwen2.5-14B @ 4-bit: ~9 GB")
print("  Reranker (XLM-R):    ~1 GB")
print("  Kaggle T4 total:      15 GB")
print("  ➜ Reviewer LoRA (Exp4/6) load TRÊN TOP Qwen đã có sẵn → không load lại base model")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 4: Load Qwen2.5-14B MỘT LẦN DUY NHẤT vào RAM/VRAM
# (4-bit NF4 quantization: 28GB → ~9GB VRAM)
# ═══════════════════════════════════════════════════════════════
import sys
sys.path.insert(0, "/kaggle/working")
from src.models.llm_server import LLMClient

print(f"🚀 Loading Qwen2.5-14B from: {QWEN_MODEL_PATH}")
print("⏳ Ước tính: 1-2 phút để load + quantize...")

SHARED_LLM = LLMClient(
    model_name=QWEN_MODEL_PATH,
    backend="transformers",
    quantization="4bit",   # NF4 double quant: 28GB → ~9GB VRAM
    temperature=0.7,
    max_tokens=4096,
)

print("\n✅ Model đã load xong vào VRAM!")
print("   SHARED_LLM sẽ được dùng chung cho TẤT CẢ 6 experiments.")
print("   Không load lại — tiết kiệm thời gian và VRAM.")

# Kiểm tra GPU memory
try:
    import torch
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved  = torch.cuda.memory_reserved()  / 1024**3
    print(f"\n   GPU VRAM used: {allocated:.1f} GB allocated, {reserved:.1f} GB reserved")
except:
    pass

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 5: Patch YAML configs với đường dẫn Kaggle thực tế
# ═══════════════════════════════════════════════════════════════
import yaml, os

def patch_config(config_path, **overrides):
    """Override nested YAML keys dạng 'a.b.c' = value."""
    if not os.path.exists(config_path):
        print(f"⚠ Không tìm thấy: {config_path}")
        return
    with open(config_path, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f) or {}
    for dotkey, val in overrides.items():
        keys = dotkey.split(".")
        node = cfg
        for k in keys[:-1]:
            node = node.setdefault(k, {})
        node[keys[-1]] = val
    with open(config_path, "w", encoding="utf-8") as f:
        yaml.dump(cfg, f, default_flow_style=False, allow_unicode=True)
    print(f"  ✅ {os.path.basename(config_path)}")

print("Patching experiment configs...")

# Exp1–3: không có fine-tuned components
# (llm config trỏ về Groq trong YAML nhưng SHARED_LLM override hết → không cần patch)

# Exp4: Reviewer LoRA
patch_config(
    "configs/experiments/exp4_agent_ft_reviewer.yaml",
    **{
        "fine_tuned.reviewer.enabled":     bool(REVIEWER_ADAPTER_PATH),
        "fine_tuned.reviewer.base_model":  QWEN_MODEL_PATH,
        "fine_tuned.reviewer.adapter_path": REVIEWER_ADAPTER_PATH or "reviewer_lora_v2_adapter",
    }
)

# Exp5: Reranker
patch_config(
    "configs/experiments/exp5_agent_ft_reranker.yaml",
    **{
        "fine_tuned.reranker.enabled":    bool(RERANKER_PATH),
        "fine_tuned.reranker.model_path": RERANKER_PATH or "reranker_ft",
    }
)

# Exp6: Cả hai
patch_config(
    "configs/experiments/exp6_agent_full.yaml",
    **{
        "fine_tuned.reviewer.enabled":     bool(REVIEWER_ADAPTER_PATH),
        "fine_tuned.reviewer.base_model":  QWEN_MODEL_PATH,
        "fine_tuned.reviewer.adapter_path": REVIEWER_ADAPTER_PATH or "reviewer_lora_v2_adapter",
        "fine_tuned.reranker.enabled":    bool(RERANKER_PATH),
        "fine_tuned.reranker.model_path": RERANKER_PATH or "reranker_ft",
    }
)

print("\n✅ Tất cả configs đã được cập nhật")
if not REVIEWER_ADAPTER_PATH:
    print("⚠ Exp4/6: reviewer FT disabled (không tìm thấy adapter)")
if not RERANKER_PATH:
    print("⚠ Exp5/6: reranker FT disabled (không tìm thấy model)")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 6: Chạy cuốn chiếu cụm 2 câu (BẢN ĐÃ SỬA LỖI OBJECT)
# ═══════════════════════════════════════════════════════════════
import sys, os, time, random, shutil
sys.path.insert(0, "/kaggle/working")

from scripts.generate_raw_data import run_experiment, DEFAULT_CONFIGS
from src.evaluation.benchmarks.loaders import HotpotQALoader

# ── 🎯 CẤU HÌNH CỤM CÂU HỎI (THAY ĐỔI THEO TỪNG LƯỢT CHẠY) ──────
# Lượt này chạy Câu 3 và Câu 4 -> tương ứng với Index 2 và 4 trong code
START_IDX = 13  
END_IDX   = 14 

OUTPUT_DIR = "data/raw_outputs"
TMP_DIR    = "data/raw_outputs_tmp"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1. Load kho 15 câu gốc
all_questions = HotpotQALoader().load(n_samples=15)

# 2. Cắt lấy đúng cụm 2 câu mục tiêu
questions_batch = all_questions[START_IDX:END_IDX]

print(f"📦 Tổng kho dữ liệu gốc: {len(all_questions)} câu.")
print(f"🚀 Lượt này tiến hành chạy từ Câu số {START_IDX + 1} đến Câu số {END_IDX}")
print(f"📝 Danh sách câu hỏi lượt này:\n")

for i, q in enumerate(questions_batch):
    # 🛠️ SỬA LỖI CHÍ MẠNG: Lấy thuộc tính dạng Object (.question) thay vì Dict
    q_text = getattr(q, 'question', str(q))
    print(f"  [{START_IDX + i + 1}] {q_text[:70]}...")

# ── Chạy tuần tự qua 6 Experiments ─────────────────────────────
results = {}
for cfg_path in DEFAULT_CONFIGS:
    if not os.path.exists(cfg_path):
        print(f"⚠ Config không tìm thấy, bỏ qua: {cfg_path}")
        continue
        
    exp_name = os.path.basename(cfg_path).replace(".yaml", "")
    print(f"\n⚙️ [EXECUTE] Đang xử lý cấu hình: {exp_name}")
    
    # Tạo thư mục tạm sạch sẽ
    if os.path.exists(TMP_DIR):
        shutil.rmtree(TMP_DIR)
    os.makedirs(TMP_DIR, exist_ok=True)
    
    # Bùa chống ban: Nghỉ giải lao ngẫu nhiên trước khi đổi cấu hình
    time.sleep(random.uniform(3.0, 6.0))
    
    # Gọi hàm xử lý cốt lõi trên tập câu hỏi nhỏ
    tmp_jsonl_path = run_experiment(
        config_path=cfg_path,
        questions=questions_batch,
        output_dir=TMP_DIR,
        shared_llm_client=SHARED_LLM,
    )
    
    # Định dạng tên file lưu trữ độc lập theo cụm index
    final_jsonl_path = os.path.join(OUTPUT_DIR, f"{exp_name}_batch_{START_IDX}_{END_IDX}.jsonl")
    
    if tmp_jsonl_path and os.path.exists(tmp_jsonl_path):
        if os.path.exists(final_jsonl_path):
            os.remove(final_jsonl_path)
        shutil.move(tmp_jsonl_path, final_jsonl_path)
        results[exp_name] = final_jsonl_path
        print(f"  📥 Xuất file batch thành công -> {final_jsonl_path}")
    else:
        print(f"  ❌ Thất bại: Không thu được kết quả từ hệ thống.")

# Dọn dẹp thư mục tạm
if os.path.exists(TMP_DIR):
    shutil.rmtree(TMP_DIR)

# ── Tóm tắt sản lượng thu hoạch ─────────────────────────────────
print("\n" + "="*60)
print(f"🎉 BATCH TỪ CÂU {START_IDX + 1} ĐẾN {END_IDX} HOÀN THÀNH AN TOÀN!")
for exp, path in results.items():
    n = sum(1 for _ in open(path)) if os.path.exists(path) else 0
    print(f"  {exp}: {n} records → {path}")
print("="*60)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 7: Tính metrics từ raw JSONL (SỬA ĐỔI: DÙNG LOCAL QWEN 14B JUDGE)
# ═══════════════════════════════════════════════════════════════
import json, os
import pandas as pd
from src.evaluation.metrics import (
    answer_f1, citation_precision, ndcg_at_k, mrr,
    is_chunk_relevant, is_evidence_sufficient, reviewer_accuracy, reviewer_f1
)
from src.evaluation.judge import LLMJudge
import src.models.llm_server

# 1. Bỏ qua hoàn toàn việc load Groq Keys
print("🧠 Cấu hình Trọng tài: Sử dụng LOCAL Qwen-14B (SHARED_LLM)...")

# 2. Gán thẳng bộ não chung SHARED_LLM vào làm Judge (Không gọi API ngoài)
# Để làm trọng tài chuẩn xác, ta ép temperature về 0.0 để kết quả chấm điểm nhất quán
SHARED_LLM.temperature = 0.0 
judge = LLMJudge(llm_client=SHARED_LLM)

# Ép thêm prompt để Qwen 14B xuất JSON cực kỳ nghiêm ngặt, tránh lỗi parse chữ
RUBRIC = (
    "You are a strict academic reviewer. You must respond with a JSON object ONLY.\\n"
    "Score from 1.0 to 5.0 based on these exact criteria:\\n"
    "COMPREHENSIVENESS: 1.0=1-3 sentences; 3.0=basic paragraphs; 5.0=multi-section report with tables.\\n"
    "DEPTH: 1.0=generic/fluffy; 3.0=mentions facts but shallow; 5.0=precise metrics, authors, mechanics."
)

# ── [GIỮ NGUYÊN HOẶC DÙNG CODES MAPPED TỰ ĐỘNG TỪ LẦN TRƯỚC] ──
raw_dir = "data/raw_outputs"
all_results = []

# Tự động xây dựng questions_map trực tiếp từ các bản ghi kết quả thô để tránh lỗi NameError
questions_map = {}
class MetaQuestion:
    def __init__(self, q_id, text, supporting_facts=None):
        self.id = q_id
        self.question = text
        self.supporting_facts = supporting_facts or []

if os.path.exists(raw_dir):
    for jsonl_file in os.listdir(raw_dir):
        if jsonl_file.endswith(".jsonl") and not jsonl_file.startswith("all_metrics"):
            for line in open(os.path.join(raw_dir, jsonl_file)):
                rec = json.loads(line)
                q_id = rec.get("question_id")
                if q_id and q_id not in questions_map:
                    s_facts = rec.get("supporting_facts", [])
                    questions_map[q_id] = MetaQuestion(q_id, rec.get("question", ""), s_facts)

print(f"✅ Đã thiết lập bản đồ đối chiếu cho {len(questions_map)} câu hỏi.")

# ── DƯỚI ĐÂY LÀ VÒNG LẶP ĐỌC FILE VÀ TÍNH METRICS (GIỮ NGUYÊN CODE CŨ CỦA BẠN) ──
for jsonl_file in sorted(os.listdir(raw_dir)):
    if not jsonl_file.endswith(".jsonl") or jsonl_file.startswith("all_metrics"):
        continue
    exp_name = jsonl_file.replace("_raw.jsonl", "")
    print(f"\n📊 {exp_name}")

    records = [json.loads(l) for l in open(os.path.join(raw_dir, jsonl_file))]

    for rec in records:
        q_id    = rec["question_id"]
        q_obj   = questions_map.get(q_id)
        supporting_facts = q_obj.supporting_facts if q_obj else []
        report  = rec.get("final_report", "")
        gt      = rec.get("ground_truth_answer", "")
        cits    = rec.get("citations", [])
        evs     = rec.get("collected_evidence", [])
        success = rec.get("success", False)
        decisions = rec.get("reviewer_decisions", [])

        f1    = answer_f1(report, gt) if report else 0.0
        cit_p = 0.0
        try:
            cit_p = citation_precision(report_body=report, citations=cits,
                                       collected_evidence=evs, judge=judge) or 0.0
        except Exception as e:
            print(f"  ⚠ citation_precision: {e}")

        # Tính ndcg và mrr
        ndcg_val = 0.0
        mrr_val = 0.0
        if supporting_facts:
            sorted_evidence = sorted(evs, key=lambda x: x.get("score", 0.0), reverse=True)
            relevance_scores = [is_chunk_relevant(ev.get("text", ""), supporting_facts) for ev in sorted_evidence]
            ndcg_val = ndcg_at_k(relevance_scores, k=5)
            mrr_val = mrr(relevance_scores)

        # Tính reviewer accuracy và F1
        rev_acc = 1.0
        rev_f1 = 1.0
        if decisions and supporting_facts:
            reviewer_preds = [d["sufficient"] for d in decisions]
            reviewer_gts = [is_evidence_sufficient(d["collected_evidence"], supporting_facts) for d in decisions]
            rev_acc = reviewer_accuracy(reviewer_preds, reviewer_gts)
            rev_f1 = reviewer_f1(reviewer_preds, reviewer_gts)

        comp = depth = overall = 1.0
        if report and success:
            try:
                jr      = judge.judge_report(report, rec["question"], RUBRIC)
                comp    = float(jr.comprehensiveness)
                depth   = float(jr.depth_of_research)
                overall = float(jr.overall_score)
            except Exception as e:
                print(f"  ⚠ judge error: {e}")

        all_results.append({
            "experiment":              exp_name,
            "question_id":             q_id,
            "answer_f1":               round(f1, 4),
            "citation_precision":      round(cit_p, 4),
            "judge_comprehensiveness": round(comp, 3),
            "judge_depth":             round(depth, 3),
            "judge_overall":           round(overall, 3),
            "ndcg_at_5":               round(ndcg_val, 4),
            "mrr":                     round(mrr_val, 4),
            "reviewer_accuracy":        round(rev_acc, 4),
            "reviewer_f1":              round(rev_f1, 4),
            "step_count":              rec.get("step_count", 0),
            "success":                 int(success),
        })
        print(f"  {q_id}: F1={f1:.3f} | CiteP={cit_p:.3f} | Judge={overall:.2f} | ✓={success}")

df_all = pd.DataFrame(all_results)
df_all.to_csv("data/raw_outputs/all_metrics_detail.csv", index=False)
print("\n✅ Saved: data/raw_outputs/all_metrics_detail.csv")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 8: Bảng so sánh Ablation + Biểu đồ
# ═══════════════════════════════════════════════════════════════
import pandas as pd, os
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

df_all = pd.read_csv("data/raw_outputs/all_metrics_detail.csv")

agg = df_all.groupby("experiment").agg(
    task_success_rate=("success", "mean"),
    answer_f1=("answer_f1", "mean"),
    citation_precision=("citation_precision", "mean"),
    judge_comprehensiveness=("judge_comprehensiveness", "mean"),
    judge_depth=("judge_depth", "mean"),
    judge_overall=("judge_overall", "mean"),
    ndcg_at_5=("ndcg_at_5", "mean"),
    mrr=("mrr", "mean"),
    reviewer_accuracy=("reviewer_accuracy", "mean"),
    reviewer_f1=("reviewer_f1", "mean"),
    avg_steps=("step_count", "mean"),
    n_questions=("question_id", "count"),
).round(4).reset_index()

EXP_LABELS = {
    "exp1_vanilla_llm":       "Exp1: Vanilla LLM",
    "exp2_rag_baseline":      "Exp2: RAG",
    "exp3_agent_base":        "Exp3: Agent",
    "exp4_agent_ft_reviewer": "Exp4: +Reviewer FT",
    "exp5_agent_ft_reranker": "Exp5: +Reranker FT",
    "exp6_agent_full":        "Exp6: Full System",
}
order = list(EXP_LABELS.keys())
agg["Config"]   = agg["experiment"].map(EXP_LABELS).fillna(agg["experiment"])
agg["sort_key"] = agg["experiment"].map({k: i for i, k in enumerate(order)}).fillna(99)
agg = agg.sort_values("sort_key").drop(columns=["sort_key", "experiment"])

cols = ["Config", "task_success_rate", "answer_f1", "citation_precision",
        "judge_comprehensiveness", "judge_depth", "judge_overall",
        "ndcg_at_5", "mrr", "reviewer_accuracy", "reviewer_f1", "avg_steps"]
agg = agg[[c for c in cols if c in agg.columns]]

agg.to_csv("logs/experiments/ablation_comparison.csv", index=False)
agg.to_json("logs/experiments/ablation_comparison.json", orient="records", indent=2)

print("🏆 ABLATION STUDY — Autonomous AI Researcher (Qwen2.5-14B local)")
print("=" * 110)
print(agg.to_string(index=False))
print("=" * 110)

# ── Bar charts ──────────────────────────────────────────────
short = ["Vanilla", "RAG", "Agent", "+Rev.FT", "+Rk.FT", "Full"]
labels = short[:len(agg)]
metrics_plot = [
    ("answer_f1",           "Answer F1",             "#4C9BE8"),
    ("citation_precision",  "Citation Precision",    "#F4845F"),
    ("judge_overall",       "LLM-Judge Overall",     "#6BCB77"),
    ("task_success_rate",   "Task Success Rate",     "#FFD93D"),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle("Ablation Study — Autonomous AI Researcher (Qwen2.5-14B)",
             fontsize=15, fontweight="bold")

for ax, (col, title, color) in zip(axes.flat, metrics_plot):
    if col not in agg.columns:
        ax.set_visible(False); continue
    vals = agg[col].fillna(0).tolist()
    bars = ax.bar(labels[:len(vals)], vals, color=color, edgecolor="white", linewidth=1.2)
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_ylim(0, max(vals) * 1.3 if max(vals) > 0 else 1.0)
    ax.grid(axis="y", alpha=0.3)
    ax.spines[["top", "right"]].set_visible(False)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f"{v:.3f}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.savefig("logs/experiments/ablation_chart.png", dpi=150, bbox_inches="tight")
plt.show()

print("\n📄 logs/experiments/ablation_comparison.csv")
print("📊 logs/experiments/ablation_chart.png")